# 👩‍⚕️ Lecture 5 – Data 100, Spring 2026

[Acknowledgments Page](https://ds100.org/sp26/acks/)

In [1]:
import numpy as np
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns
#%matplotlib inline
#plt.rcParams['figure.figsize'] = (12, 9)

sns.set()
sns.set_context('talk')
np.set_printoptions(threshold=20, precision=2, suppress=True)
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)
# Use 5 decimal places instead of scientific notation
pl.Config.set_float_precision(5)

# Silence some spurious seaborn warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## Loading Data

In [2]:
import urllib.request
import os.path
import zipfile

data_url = "https://www.ssa.gov/oact/babynames/state/namesbystate.zip"
local_filename = "data/babynamesbystate.zip"
if not os.path.exists(local_filename): # If the data exists don't download again
    with urllib.request.urlopen(data_url) as resp, open(local_filename, 'wb') as f:
        f.write(resp.read())

zf = zipfile.ZipFile(local_filename, 'r')

ca_name = 'STATE.CA.TXT'
field_names = ['State', 'Sex', 'Year', 'Name', 'Count']
with zf.open(ca_name) as fh:
    babynames = pl.read_csv(fh, has_header=False, new_columns=field_names)

babynames.sample(3)

State,Sex,Year,Name,Count
str,str,i64,str,i64
"""CA""","""F""",1962,"""Hallie""",10
"""CA""","""F""",1994,"""Abigal""",6
"""CA""","""F""",1928,"""Mary""",1785


## 💁 Gender-Neutrality in Names

In [3]:
def gender_neutrality(counts):
    """Given a column of counts for different genders, computes how gender-neutral the name is"""
    # How many different ways can you find to calculate the same metric?
    return 2 * (0.5 - (0.5 - counts.first() / counts.sum()).abs())

In [4]:
neutralities_06 = (
    babynames.filter(pl.col('Year') == 2006)
    .group_by('Name')
    .agg(gender_neutrality(pl.col('Count')).alias('Gender neutrality'))
)
neutralities_06.sort('Gender neutrality', descending=True).head(10)

Name,Gender neutrality
str,f64
"""Issa""",1.00000
"""Jaelin""",1.00000
"""Nikita""",1.00000
"""Kerry""",1.00000
"""Blair""",1.00000
"""Dakoda""",1.00000
"""Harley""",0.98701
"""Robin""",0.97561
"""Riley""",0.97503


In [5]:
# Gender-neutrality of each name within each year
neut_yearly = (
    babynames.group_by(['Name', 'Year'])
    .agg(gender_neutrality(pl.col('Count')).alias('Gender neutrality'))
)
neut_yearly

Name,Year,Gender neutrality
str,i64,f64
"""Demetra""",2004,0.00000
"""Serenity""",1982,0.00000
"""Khang""",1989,0.00000
"""Tou""",2000,0.00000
"""Akshaj""",2013,0.00000
"""Addalynn""",2019,0.00000
"""Gareth""",1986,0.00000
"""Sophia""",2021,0.00000
"""Lilyana""",2020,0.00000


In [6]:
most_neutral_name_by_year = (
    neut_yearly
    .sort('Gender neutrality', descending=True)
    .group_by('Year').head(1)
    .sort('Year')
)
most_neutral_name_by_year

Year,Name,Gender neutrality
i64,str,f64
1910,"""Lee""",0.63158
1911,"""Merle""",0.90909
1912,"""Merle""",0.85714
1913,"""Gene""",0.71429
1914,"""Beverly""",0.88889
1915,"""Merle""",0.92857
1916,"""Merle""",0.96552
1917,"""Trinidad""",0.85714
1918,"""Refugio""",1.00000


This gives us the most gender-neutral name for each year, but we can see by inspecting it that many of these names are rare. What if we're only interested in more common names?

We need to use the counts.

In [7]:
yearly_name_counts = babynames.group_by(['Name', 'Year']).agg(pl.col('Count').sum())
yearly_name_counts.head(4)

Name,Year,Count
str,i64,i64
"""Gisel""",1998,31
"""Johathan""",1992,16
"""Gordon""",1925,128
"""Jorgeluis""",1990,9


In [8]:
counts_and_neutrality = yearly_name_counts.join(
    neut_yearly,
    left_on=['Name', 'Year'], right_on=['Name', 'Year']
)
counts_and_neutrality

Name,Year,Count,Gender neutrality
str,i64,i64,f64
"""Demetra""",2004,5,0.00000
"""Serenity""",1982,11,0.00000
"""Khang""",1989,7,0.00000
"""Tou""",2000,6,0.00000
"""Akshaj""",2013,8,0.00000
"""Addalynn""",2019,7,0.00000
"""Gareth""",1986,11,0.00000
"""Sophia""",2021,1816,0.00000
"""Lilyana""",2020,38,0.00000


In [9]:
most_neutral_name_by_year = (
    counts_and_neutrality.filter(pl.col('Count') > 500)
    .sort('Gender neutrality', descending=True)
    .group_by('Year').head(1)
    .sort('Year')
)
most_neutral_name_by_year

Year,Name,Count,Gender neutrality
i64,str,i64,f64
1912,"""Mary""",534,0.00000
1913,"""William""",528,0.00000
1914,"""Mary""",773,0.00000
1915,"""Mary""",1003,0.00997
1916,"""Robert""",903,0.00000
1917,"""Mary""",1154,0.00867
1918,"""Mary""",1260,0.01429
1919,"""Frank""",594,0.01684
1920,"""James""",840,0.01429


In [10]:
sns.lineplot(most_neutral_name_by_year.to_pandas(), x='Year', y='Gender neutrality')

<Axes: xlabel='Year', ylabel='Gender neutrality'>

In [11]:
# What happened around 1940?
most_neutral_name_by_year.filter(
    (pl.col('Year') > 1935) &
    (pl.col('Year') < 1950)
)

Year,Name,Count,Gender neutrality
i64,str,i64,f64
1936,"""Marilyn""",553,0.02532
1937,"""Margaret""",575,0.02783
1938,"""Jerry""",559,0.05725
1939,"""Jerry""",603,0.05638
1940,"""Jerry""",660,0.06667
1941,"""Jerry""",748,0.09893
1942,"""Jerry""",911,0.07903
1943,"""Terry""",584,0.54110
1944,"""Terry""",591,0.52792


## 🦠 Flu in the United States

What can we say about flu in the United States?

### 📖 Reading CSVs

The flu data contains several CSV files, located in `data/flu/`. We can explore them in many ways:

1. Using the JupyterLab explorer tool (read-only!).
2. Opening the CSV in DataHub, or Excel, or Google Sheets, etc.
3. Opening the file in Python with `open()`
4. With `Polars`, using `pl.read_csv()`

<br>


---

### 🧭 Methods 1 and 2: Play with the data in the JupyterLab Explorer and DataHub
 To solidify the idea of a CSV as **rectangular data** (i.e., tabular data) stored as comma-separated values, let's start with the first two methods.  

 **1. Use the file browser in JupyterLab to locate one of the CSVs in `data/flu.csv`, and double-click on it.**

  **2. Right-click on the CSV in the file browser. Select `Open With` --> `Editor`. But, don't make any changes to the file!**

<br>

---

### 🐍 Method 3: Play with the data in Python

Next, we will load in the data as a Python file object and inspect a couple lines. 

With the code below, we can check out the first four lines of the CSV:

In [12]:
# Open the FLU ILINet CSV, and print the first four lines
with open("data/flu/ILINet.csv", "r") as f:
    for i, row in enumerate(f):
        print(row)
        if i >= 3: break

PERCENTAGE OF VISITS FOR INFLUENZA-LIKE-ILLNESS REPORTED BY SENTINEL PROVIDERS

REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,%UNWEIGHTED ILI,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS

HHS Regions,Region 1,2015,40,0.743302,0.684364,103,50,,133,23,13,322,134,47051

HHS Regions,Region 2,2015,40,1.03278,1.22475,547,294,,528,123,95,1587,199,129577



As expected, most of the lines are comma-separated values. But what's up with the first line?

> Why are there blank lines between each line of the CSV file?
>
> You may recall that line breaks in text files are encoded with the special newline character `\n`. 
> 
> Python's `print()` function prints each line, interpreting the `\n` at the end of each line as a newline, **and also adds an additional newline**.

We can use the `repr()` ("representation") function to return the raw string representation of each line (i.e., all special characters will be visible).

- In other words, `print()` will not interpret `\n` as a newline. Instead, it will literally print `\n`.

- Note, though, `print()` adds a newline each time it is called. Otherwise, we would have one long string below instead of four lines.

In [13]:
# Open the TB case count CSV, and print the raw representation of
# the first four lines
with open("data/flu/ILINet.csv", "r") as f:
    for i, row in enumerate(f):
        print(repr(row)) # print raw strings
        if i >= 3: break

'PERCENTAGE OF VISITS FOR INFLUENZA-LIKE-ILLNESS REPORTED BY SENTINEL PROVIDERS\n'
'REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,%UNWEIGHTED ILI,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS\n'
'HHS Regions,Region 1,2015,40,0.743302,0.684364,103,50,,133,23,13,322,134,47051\n'
'HHS Regions,Region 2,2015,40,1.03278,1.22475,547,294,,528,123,95,1587,199,129577\n'


### 🐻‍❄️ Method 4: Play with the data using `polars`

Time to use our favorite Data 100 approach: `Polars`.

In [14]:
ili = pl.read_csv("data/flu/ILINet.csv")
ili

ComputeError: found more fields than defined in 'Schema'

Consider setting 'truncate_ragged_lines=True'.

What went wrong? `Polars` builds the schema from the first line of the file, and the error tells us
that later lines carry more fields than that. The first line here is a title, not a header row.

We're ready to wrangle the data!

A reasonable first step is to **identify the row with the right header** (i.e., the row with the column names).

The `pl.read_csv()` function ([documentation](https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html)) has a convenient `skip_rows` parameter for skipping the lines that come before the header row:

In [15]:
# Try again, this time skipping the title line so the header is read from the second line
ili = pl.read_csv("data/flu/ILINet.csv", skip_rows=1)
ili.head()

REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,%UNWEIGHTED ILI,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS
str,str,i64,i64,f64,f64,i64,i64,str,i64,i64,i64,i64,i64,i64
"""HHS Regions""","""Region 1""",2015,40,0.74330,0.68436,103,50,null,133,23,13,322,134,47051
"""HHS Regions""","""Region 2""",2015,40,1.03278,1.22475,547,294,null,528,123,95,1587,199,129577
"""HHS Regions""","""Region 3""",2015,40,1.21780,1.24313,401,419,null,625,144,81,1670,280,134338
"""HHS Regions""","""Region 4""",2015,40,1.01464,1.15781,486,231,null,613,99,75,1504,299,129900
"""HHS Regions""","""Region 5""",2015,40,1.04259,1.17723,384,238,null,444,159,103,1328,284,112807


#### 🔎 Granularity of records

What is the granularity of each record in our flu dataset? Region? Week? Age range?

The answer is that each row represents a unique combination of week, year, and region.

We should probably start by making some **visualizations**: a picture is always worth a thousand words.

### 📈 Time to visualize! (Almost)

#### Preprocessing and column manipulation

...except we can't just yet. Since we have one row per year/week, we probably want to use a line graph. But, we need one column we can use for the x-axis in our graph. How do we convert a year column + week column into a single date column?

In [16]:
# Week numbers count from the first Monday of the year, so build that Monday first, step forward
# one week per WEEK, then add 6 days to land on the Sunday that closes the week.
# .dt.weekday() numbers Monday as 1 and Sunday as 7.
jan_1 = pl.date(pl.col('YEAR'), 1, 1)
first_monday = jan_1 + pl.duration(days=(8 - jan_1.dt.weekday()) % 7)

ili = ili.with_columns(
    (first_monday + pl.duration(weeks=pl.col('WEEK') - 1, days=6)).alias('week_start')
)
ili.sample(3)

REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,%UNWEIGHTED ILI,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS,week_start
str,str,i64,i64,f64,f64,i64,i64,str,i64,i64,i64,i64,i64,i64,date
"""HHS Regions""","""Region 8""",2021,16,0.81154,0.79666,155,156,null,200,54,47,612,208,76821,2021-04-25
"""HHS Regions""","""Region 6""",2023,1,4.96587,4.68267,1383,2082,null,1881,1053,805,7204,264,153844,2023-01-08
"""HHS Regions""","""Region 4""",2024,8,4.48665,4.40500,4931,6989,null,8972,2688,2426,26006,938,590374,2024-02-25


We can access different attributes of the new column through the `.dt` namespace:

In [17]:
ili['week_start'].dt.year().head()

week_start
i32
2015
2015
2015
2015
2015
2015
2015
2015
2015


What is the type of this new column?

In [18]:
ili['week_start'].dtype

Date

`Date` is the `Polars` type for a calendar day.

- A column that also carries a time of day has type `Datetime`, which records a time unit such as microseconds

Under the hood, a `Date` is an integer counting the number of **days** since 1/1/1970 UTC.

#### Visualization

Don't worry too much about the exact code that generates the graph: we'll talk more about it next week.

In [19]:
# Visualize number of cases for babies and small children
f, ax = plt.subplots(1, 1, figsize=(12, 7))
sns.lineplot(ili.to_pandas(), x='week_start', y='AGE 0-4', hue='REGION', ax=ax);

In [20]:
# Visualize overall prevalence of flu
f, ax = plt.subplots(1, 1, figsize=(12, 7))
sns.lineplot(ili.to_pandas(), x='week_start', y='% WEIGHTED ILI', hue='REGION', ax=ax);

At this point, stop and think about any interesting observations or questions you have about this data.

<br/><br/>

---

**Instructor note: Return to the slides!**


---
<br/><br/>

## Load vaccination data

This dataframe contains vaccination numbers by each flu season (starting in July and ending the following June).

Take a close look at the output, and make sure you understand what happens between 2023-06-01 and 2023-07-01!

In [21]:
vax = pl.read_csv('data/flu/monthly_child_flu_vaccination.csv')
vax = vax.with_columns(
    pl.col('month_dt').str.to_date(),
    rate=pl.col('Numerator') / pl.col('Population'),
)
vax.head(14)

HHS Region,month_dt,Numerator,Population,rate
str,date,f64,f64,f64
"""Region 1""",2022-07-01,17110.00000,1328581.00000,0.01288
"""Region 1""",2022-08-01,42110.00000,1328581.00000,0.03170
"""Region 1""",2022-09-01,129698.00000,1328581.00000,0.09762
"""Region 1""",2022-10-01,297855.00000,1328581.00000,0.22419
"""Region 1""",2022-11-01,430376.00000,1328581.00000,0.32394
"""Region 1""",2022-12-01,508781.00000,1328581.00000,0.38295
"""Region 1""",2023-01-01,545783.00000,1328581.00000,0.41080
"""Region 1""",2023-02-01,563456.00000,1328581.00000,0.42410
"""Region 1""",2023-03-01,575885.00000,1328581.00000,0.43346


<br/><br/>

---

## 👥 Joining ILI counts with vaccination data

Time to `join` our datasets!

In [22]:
# Show the three tables that we are going to join.
# To keep things simple, let's just look at the last two rows of each df.
display(ili.tail(2))
display(vax.tail(2))

REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,%UNWEIGHTED ILI,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS,week_start
str,str,i64,i64,f64,f64,i64,i64,str,i64,i64,i64,i64,i64,i64,date
"""HHS Regions""","""Region 9""",2026,3,4.85718,4.62364,2045,5863,null,4914,2552,3004,18378,402,397479,2026-01-25
"""HHS Regions""","""Region 10""",2026,3,5.06699,4.91184,1462,1946,null,3526,700,793,8427,332,171565,2026-01-25


HHS Region,month_dt,Numerator,Population,rate
str,date,f64,f64,f64
"""Region 9""",2025-11-01,1416002.00000,8216613.00000,0.17233
"""Region 9""",2025-12-01,1686327.00000,8216613.00000,0.20523


We need to have a month column in ILI so that we can join them. How do we convert our `week_start` column to the corresponding month? This is a great time to do a Google search and/or ask an LLM!

In [23]:
ili = ili.with_columns(pl.col("week_start").dt.truncate("1mo").alias("month"))

In [24]:
ili_vax = ili.join(
    vax,
    left_on=['month', 'REGION'],
    right_on=['month_dt', 'HHS Region']
)
ili_vax.head()

REGION TYPE,REGION,YEAR,WEEK,% WEIGHTED ILI,%UNWEIGHTED ILI,AGE 0-4,AGE 25-49,AGE 25-64,AGE 5-24,AGE 50-64,AGE 65,ILITOTAL,NUM. OF PROVIDERS,TOTAL PATIENTS,week_start,month,Numerator,Population,rate
str,str,i64,i64,f64,f64,i64,i64,str,i64,i64,i64,i64,i64,i64,date,date,f64,f64,f64
"""HHS Regions""","""Region 1""",2022,26,0.93066,1.02061,440,356,null,346,166,211,1519,232,148833,2022-07-03,2022-07-01,17110.00000,1328581.00000,0.01288
"""HHS Regions""","""Region 2""",2022,26,2.80148,2.49609,1756,630,null,1124,278,297,4085,161,163656,2022-07-03,2022-07-01,699.00000,1982009.00000,0.00035
"""HHS Regions""","""Region 3""",2022,26,1.38848,1.62687,1347,883,null,929,397,360,3916,371,240707,2022-07-03,2022-07-01,284.00000,4301556.00000,0.00007
"""HHS Regions""","""Region 4""",2022,26,2.29010,2.47071,4320,3555,null,3430,1469,1370,14144,928,572466,2022-07-03,2022-07-01,2268.00000,13203279.00000,0.00017
"""HHS Regions""","""Region 5""",2022,26,1.15038,1.05771,1001,678,null,817,331,349,3176,610,300270,2022-07-03,2022-07-01,662.00000,11492529.00000,0.00006


In [25]:
f, ax = plt.subplots(1, 1, figsize=(10, 7))
sns.scatterplot(ili_vax.to_pandas(), x='ILITOTAL', y='rate', alpha=0.3, hue='REGION', ax=ax)

<Axes: xlabel='ILITOTAL', ylabel='rate'>

What are some limitations of this graph?